# DEC5 000973: source-base transport and explicit multi-camera blending
One temporal frame, seven transport variants (19 held-anchor outputs) and two mixture variants (six outputs). All complete native gates fail. This companion verifies comparison integrity; it does not convert a small face-metric gain into a neck/tube repair.


In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair')
families = ['base_transport_control', 'visible_source_blend_control']
results = {name: json.loads((root/name/'findings.json').read_text()) for name in families}
def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()
for name, result in results.items():
    assert result['temporal_frame_count'] == 1
    assert result['audit_status'] == 'comparison_integrity_pass_candidate_quality_fail'
    assert result['visual_pass'] == 0
    keys = [(r['anchor'], r['variant']) for r in result['rows']]
    assert len(keys) == len(set(keys)) == result['variant_anchor_count']
    assert result['visual_fail'] == len(keys)
assert results[families[0]]['variant_anchor_count'] == 19
assert results[families[1]]['variant_anchor_count'] == 6


In [2]:
checked = {}
for name, result in results.items():
    for path, expected in result['retained_hashes'].items():
        # Historical scripts are resolved to exact saved snapshots by the audit.
        actual = checked.setdefault(path, sha(path))
        assert actual == expected, path
    print(name, 'retained inputs/outputs verified:', len(result['retained_hashes']))
assert results[families[0]]['depth_maps_verified'] == 62
print('All retained hashes match; original 62 geometric maps remain bound.')


base_transport_control retained inputs/outputs verified: 336
visible_source_blend_control retained inputs/outputs verified: 156
All retained hashes match; original 62 geometric maps remain bound.


In [3]:
baseline = json.loads((root/'base_transport_control/F/metrics_baseline/metrics.json').read_text())
common = ['ground_truth_sha256', 'face_polygons_sha256', 'face_bbox_xyxy', 'protocol', 'metric_domain']
for variant in ['full_rgb', 'low_band']:
    metrics = json.loads((root/'visible_source_blend_control/F'/('metrics_'+variant)/'metrics.json').read_text())
    assert all(metrics[k] == baseline[k] for k in common)
    assert all(np.isfinite(metrics[k]) for k in ['face_psnr', 'face_ssim', 'face_lpips'])
    assert not any(k.startswith(('loss', 'full_frame', 'actor_psnr', 'room_psnr')) for k in metrics)
    print(variant, {k: metrics[k] for k in ['face_psnr', 'face_ssim', 'face_lpips']},
          'LPIPS / matched baseline:', metrics['face_lpips']/baseline['face_lpips'])
assert baseline['protocol']['candidate_surface_mask'] is False
assert baseline['protocol']['prediction_used_for_roi_selection'] is False
print('Study face bbox:', baseline['face_bbox_xyxy'], '; most neck/hand is outside this metric ROI.')


full_rgb {'face_psnr': 29.767833709716797, 'face_ssim': 0.9321650266647339, 'face_lpips': 0.11515151709318161} LPIPS / matched baseline: 2.399528489731028
low_band {'face_psnr': 29.149364471435547, 'face_ssim': 0.8906574249267578, 'face_lpips': 0.04616355150938034} LPIPS / matched baseline: 0.9619565580216131
Study face bbox: [700, 388, 1165, 726] ; most neck/hand is outside this metric ROI.


In [4]:
from PIL import Image
for anchor in ['F', 'J', 'L']:
    directory = root/'visible_source_blend_control'/anchor
    request = json.loads((directory/'request.json').read_text())
    assert request['source_rgb_averaging'] and not request['uses_eval_rgb']
    assert request['geometry_unchanged'] and request['visibility_unchanged']
    assert request['train_camera_count'] == 62 and len(set(request['sources'])) == 8
    assert not set(request['sources']) & {'F004_B005_1210O9','J004_D005_1210TA','L004_B005_12106A'}
    warps = next(Path(p).parent for p in request['input_hashes'] if Path(p).name == 'source_00.png')
    valid = np.stack([np.asarray(Image.open(warps/f'valid_{i:02d}.png')) > 0 for i in range(8)])
    weights = np.load(directory/'source_weights.npz', allow_pickle=False)['weights']
    assert weights.shape == (8,1080,1920) and np.isfinite(weights).all()
    assert (weights >= 0).all() and not weights[~valid].any()
    total = weights.sum(0)
    assert np.max(np.abs(total[total > 0]-1)) < 3e-7
    print(anchor, 'visible-only normalized weights verified')
print('No temporal promotion. Native verdicts are saved separately and cannot be inferred from PSNR/SSIM.')


F visible-only normalized weights verified


J visible-only normalized weights verified


L visible-only normalized weights verified
No temporal promotion. Native verdicts are saved separately and cannot be inferred from PSNR/SSIM.


## Interpretation
Full RGB averaging visibly softens the patch and fine details; higher PSNR/SSIM accompanies worse LPIPS. Low-band averaging preserves detail and slightly improves face metrics, but leaves the conspicuous neck/tube seam. Transport variants likewise fail native gates. This is evidence about one time and three held cameras, not a completed every-40th-time campaign or continuous fly-through.
